Environment Setup & Library Installation

In [ ]:
!pip install -q -U transformers accelerate
!pip install -q sentence-transformers
!pip install -q newsapi-python
!pip install -q Pillow requests tqdm
!pip install -q wikipedia

IMPORTING LIBRARIES

In [ ]:
import transformers, sentence_transformers, torch
import os, re, json, time, warnings
import requests
import numpy as np
import pandas as pd
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings('ignore')

Semantic Embedding Module (Sentence Transformers)

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Dataset Loading & Cleaning

In [ ]:
CSV_PATH = "/content/drive/MyDrive/CVProject/dataset_with_clip.csv"

df = pd.read_csv(CSV_PATH)

# Clean dataset
df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])
df['clean_title'] = df['clean_title'].fillna(df['title'])
df = df.dropna(subset=['clean_title']).reset_index(drop=True)

print("Dataset loaded")
print("Rows:", len(df))
print("Columns:", list(df.columns))

print("\nLabel breakdown (2-way):")
print(df['2_way_label'].value_counts().rename(index={0:'REAL', 1:'FAKE'}))

LOAD BLIP MODEL

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
device = "cuda" if torch.cuda.is_available() else "cpu"

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

In [ ]:
!pip install -q -U google-generativeai

SET API KEYS

In [ ]:
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"
os.environ["NEWS_API_KEY"] = "YOUR_API_KEY"

In [ ]:
API_KEY = os.getenv("GEMINI_API_KEY")

url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={API_KEY}"

data = {
  "contents": [{
    "parts": [{"text": "Say hello"}]
  }]
}

response = requests.post(url, json=data)

print(response.json())

INIT NEWS API

In [ ]:
from newsapi import NewsApiClient
newsapi = NewsApiClient(api_key=os.getenv("NEWS_API_KEY"))

Real-Time News Evidence Retrieval Module

In [ ]:
def get_news_evidence(query):
    try:
        trusted_sources = 'bbc-news,the-hindu,cnn'

        articles = newsapi.get_everything(
            q=query,
            sources=trusted_sources,
            language='en',
            page_size=3
        )

        if articles and len(articles['articles']) >= 2:
            selected_articles = articles['articles']
        else:
            fallback_sources = 'reuters,associated-press,al-jazeera-english,independent,dw-news,abc-news'

            fallback = newsapi.get_everything(
                q=query,
                sources=fallback_sources,
                language='en',
                page_size=3
            )

            selected_articles = articles.get('articles', []) + fallback.get('articles', [])

        return " ".join([
            f"{a['title']}. {a.get('description','')}"
            for a in selected_articles
        ])

    except:
        return "No news evidence found"

Knowledge Base Retrieval using Wikipedia

In [ ]:
import wikipedia

def get_wikipedia_evidence(query):
    try:
        # Search related pages first
        search_results = wikipedia.search(query)

        if not search_results:
            return "No Wikipedia evidence found"

        # Take top result
        top_page = search_results[0]

        # Get summary
        summary = wikipedia.summary(top_page, sentences=2)

        return summary

    except:
        return "No Wikipedia evidence found"

IMAGE CAPTIONING (BLIP)

In [ ]:
def generate_caption(image_url):

    if not image_url or str(image_url) == "nan":
        return "No image description available"

    try:
        response = requests.get(image_url, timeout=5)
        image = Image.open(BytesIO(response.content)).convert("RGB")

        inputs = blip_processor(images=image, return_tensors="pt").to(device)
        output = blip_model.generate(**inputs)

        return blip_processor.decode(output[0], skip_special_tokens=True)

    except:
        return "No image description available"

LLM Reasoning Module

In [ ]:
def llm_reasoning(claim, caption, evidence):

    api_key = os.getenv("GEMINI_API_KEY")

    url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash-lite:generateContent?key={api_key}"

    prompt = f"""
You are helping in a fake news detection task.

Claim:
{claim}

Image Description:
{caption}

Evidence:
{evidence}

Instructions:
- If the claim is clearly supported by factual evidence → Real
- If the claim is clearly false or contradicted → Fake
- If the claim is opinion-based, ambiguous, or not fully verifiable → Uncertain

Give output in this format:

Prediction:
Real / Fake / Uncertain

Justification:
Explain briefly

Confidence:
0 to 1
"""

    data = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ]
    }

    # trying few times in case API fails
    for i in range(3):
        try:
            response = requests.post(url, json=data)
            result = response.json()

            print("Attempt:", i+1)

            # if response is correct
            if "candidates" in result:
                return result.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text", "")

            # if API gives error
            if "error" in result:
                print("API error:", result["error"]["message"])

                # skip if server busy
                if "high demand" in result["error"]["message"].lower():
                    break

                time.sleep(3)

        except Exception as e:
            print("Error:", e)
            time.sleep(3)

    # fallback if nothing works
    return """Prediction:
Uncertain

Justification:
Not enough information or API issue

Confidence:
0.3
"""

LLM Output Parsing

In [ ]:

def parse_output(output):

    prediction = "Unknown"
    justification = ""
    confidence = 0.0

    try:
        lines = [line.strip() for line in output.split("\n") if line.strip()]

        for line in lines:

            if line.lower().startswith("prediction"):
                if ":" in line:
                    prediction = line.split(":", 1)[1].strip()

            elif line.lower().startswith("justification"):
                if ":" in line:
                    justification = line.split(":", 1)[1].strip()

            elif line.lower().startswith("confidence"):
                if ":" in line:
                    confidence = float(line.split(":", 1)[1].strip())

        # simple correction
        if prediction.lower() == "uncertain" and confidence > 0.7:
            confidence = 0.5

    except Exception as e:
        print("Parsing error:", e)

    return prediction, justification, confidence

Evidence Ranking [Contextual Similarity Filtering using Sentence Transformers]

In [ ]:

def retrieve_evidence(query):

    try:
        news = get_news_evidence(query)
        wiki = get_wikipedia_evidence(query)

        docs = []

        if wiki and wiki != "No Wikipedia evidence found":
            docs.append(wiki)

        if news and news != "No news evidence found":
            docs.extend(news.split(". "))

        if len(docs) == 0:
            return f"No external evidence found. Analyze based on claim: {query}"

        query_emb = embedder.encode(query, convert_to_tensor=True)
        doc_embs = embedder.encode(docs, convert_to_tensor=True)

        scores = util.cos_sim(query_emb, doc_embs)[0]

        top_k = min(3, len(docs))
        top_indices = scores.argsort(descending=True)[:top_k]

        top_docs = [docs[i] for i in top_indices]

        return " ".join(top_docs)

    except Exception as e:
        print("Retrieval error:", e)
        return "No external evidence found"

Main Pipeline Function

In [ ]:
def run_pipeline(row):

    claim = row['title']
    image = row['image_url']

    # Step 1: Caption
    caption = generate_caption(image)

    # Step 2: Evidence
    evidence = retrieve_evidence(claim)

    # Step 3: LLM
    llm_output = llm_reasoning(claim, caption, evidence)

    # Step 4: Parse
    prediction, justification, confidence = parse_output(llm_output)

    return {
        "Claim": claim,
        "Caption": caption,
        "Evidence": evidence,
        "Prediction": prediction,
        "Justification": justification,
        "Confidence": confidence
    }

TEST RUN

In [ ]:
result = run_pipeline(df.iloc[0])
print(result)

In [ ]:
result = run_pipeline(df.iloc[1])
print(result)

In [ ]:
result = run_pipeline(df.iloc[11])
print(result)

In [ ]:
result = run_pipeline(df.iloc[54])
print(result)

In [ ]:
TEST_ROWS = 100
TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/100test_run_output.csv"

from datetime import datetime

print("Running verification on 10 rows...")

# take only first 10 rows
test_df = df.head(TEST_ROWS).copy()

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        # store outputs
        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE TEST OUTPUT
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("Test run completed!")
print(f"File saved at: {TEST_SAVE_PATH}")

In [ ]:
START_ROW = 17   # change this (last processed row)
END_ROW = 40

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/test_run_output2.csv"

from datetime import datetime

print(f"Running verification from row {START_ROW} to {END_ROW}...")

# slice dataset
test_df = df.iloc[START_ROW:END_ROW].copy().reset_index(drop=True)

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("Test run completed!")
print(f"File saved at: {TEST_SAVE_PATH}")

In [ ]:
MISSING_ROW = 40

test_df = df.iloc[MISSING_ROW:MISSING_ROW+1].copy().reset_index(drop=True)

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/missing_test_run_output40.csv"

from datetime import datetime

print(f"Running verification from row {START_ROW} to {END_ROW}...")



# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("Test run completed!")
print(f"File saved at: {TEST_SAVE_PATH}")

In [ ]:
START_ROW = 61   # change this (last processed row)
END_ROW = 80

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/test_run_output4.csv"

from datetime import datetime

print(f"Running verification from row {START_ROW} to {END_ROW}...")

# slice dataset
test_df = df.iloc[START_ROW:END_ROW].copy().reset_index(drop=True)

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("Test run completed!")
print(f"File saved at: {TEST_SAVE_PATH}")

In [ ]:
MISSING_ROW = 60

test_df = df.iloc[MISSING_ROW:MISSING_ROW+1].copy().reset_index(drop=True)

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/missing_test_run_output60.csv"

from datetime import datetime

print(f"Running verification from row {START_ROW} to {END_ROW}...")



# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("Test run completed!")
print(f"File saved at: {TEST_SAVE_PATH}")

In [ ]:
START_ROW = 80   # change this (last processed row)
END_ROW = 100

TEST_SAVE_PATH = "/content/drive/MyDrive/CVProject/test_run_output5.csv"

from datetime import datetime

print(f"Running verification from row {START_ROW} to {END_ROW}...")

# slice dataset
test_df = df.iloc[START_ROW:END_ROW].copy().reset_index(drop=True)

# add required columns
test_df["llm_prediction"] = None
test_df["llm_confidence"] = None
test_df["justification"] = None
test_df["caption"] = None
test_df["evidence"] = None
test_df["response_time"] = None
test_df["timestamp"] = None

# ==============================
# LOOP
# ==============================
for i in tqdm(range(len(test_df))):

    row = test_df.iloc[i]

    try:
        start_time = time.time()

        result = run_pipeline(row)

        end_time = time.time()

        test_df.at[i, "llm_prediction"] = result["Prediction"]
        test_df.at[i, "llm_confidence"] = result["Confidence"]
        test_df.at[i, "justification"] = result["Justification"]
        test_df.at[i, "caption"] = result["Caption"]
        test_df.at[i, "evidence"] = result["Evidence"]
        test_df.at[i, "response_time"] = round(end_time - start_time, 2)
        test_df.at[i, "timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    except Exception as e:
        print(f"Error at row {i}: {e}")

        test_df.at[i, "llm_prediction"] = "Uncertain"
        test_df.at[i, "llm_confidence"] = 0.3
        test_df.at[i, "justification"] = "Pipeline failed"

    time.sleep(2)

# ==============================
# SAVE
# ==============================
test_df.to_csv(TEST_SAVE_PATH, index=False)

print("Test run completed!")
print(f" File saved at: {TEST_SAVE_PATH}")

In [ ]:
import os

files = os.listdir("/content/drive/MyDrive/CVProject")

for f in files:
    if "test_run" in f:
        print(f)

In [ ]:
import pandas as pd

# Your files in correct order
files = [
    "/content/drive/MyDrive/CVProject/test_run_output1.csv",
    "/content/drive/MyDrive/CVProject/test_run_output2.csv",
    "/content/drive/MyDrive/CVProject/missing_test_run_output40.csv",
    "/content/drive/MyDrive/CVProject/test_run_output3.csv",
    "/content/drive/MyDrive/CVProject/missing_test_run_output60.csv",
    "/content/drive/MyDrive/CVProject/test_run_output4.csv",
    "/content/drive/MyDrive/CVProject/test_run_output5.csv"
]

# Read and store all
dfs = []
for file in files:
    print(f"Loading: {file}")
    df = pd.read_csv(file)
    dfs.append(df)

# Merge in same order
merged_df = pd.concat(dfs, ignore_index=True)

# Save final file
SAVE_PATH = "/content/drive/MyDrive/CVProject/final_llm_score_dataset.csv"
merged_df.to_csv(SAVE_PATH, index=False)

print("All files merged successfully!")
print("Saved at:", SAVE_PATH)